In [1]:
# ============================================================
# 05_R_cpu_apollo_post_estimation.ipynb
# Apollo CPU post-estimation benchmark for the Swiss route-choice MXL model.
#
# Draws: 500, 1000, 2000, 5000, 10000
# Optimiser: BGW | CPU cores used: 8
# Records Apollo pre-/estimation/post-estimation time and CPU/R environment.
#
# Outputs:
# - results/benchmark_summary_apollo_bgw_cpu.xlsx
# - results/session_info_apollo_cpu.txt
# ============================================================


In [2]:
# -------------------------------------------------------------------
# 1. Packages
# -------------------------------------------------------------------
# Install only the packages needed for the Apollo CPU benchmark.

options(HTTPUserAgent = sprintf(
  "R/%s R (%s)",
  getRversion(),
  paste(
    getRversion(),
    R.version["platform"],
    R.version["arch"],
    R.version["os"]
  )
))

ubuntu_codename <- tryCatch(
  system("bash -lc '. /etc/os-release; echo ${VERSION_CODENAME}'", intern = TRUE),
  error = function(e) "jammy"
)

ppm_repo <- sprintf(
  "https://packagemanager.posit.co/cran/__linux__/%s/latest",
  ubuntu_codename[1]
)
options(repos = c(CRAN = ppm_repo))

for (pkg in c("apollo", "writexl")) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, Ncpus = parallel::detectCores())
  }
}

suppressPackageStartupMessages({
  library(apollo)
})

pkg_versions <- data.frame(
  package = c("apollo", "bgw", "writexl"),
  version = vapply(
    c("apollo", "bgw", "writexl"),
    function(pkg) as.character(packageVersion(pkg)),
    character(1)
  )
)
print(pkg_versions)


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘zoo’, ‘future’, ‘globals’, ‘SparseM’, ‘MatrixModels’, ‘miscTools’, ‘sandwich’, ‘rngWELL’, ‘truncnorm’, ‘future.apply’, ‘mcmc’, ‘quantreg’, ‘maxLik’, ‘mnormt’, ‘mvtnorm’, ‘randtoolbox’, ‘numDeriv’, ‘Deriv’, ‘matrixStats’, ‘coda’, ‘bgw’, ‘Rsolnp’, ‘MCMCpack’, ‘RcppArmadillo’, ‘RcppEigen’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



        package version
apollo   apollo   0.3.9
bgw         bgw   0.1.4
writexl writexl   2.0.1


In [3]:
# -------------------------------------------------------------------
# 2. Project paths
# -------------------------------------------------------------------

proj_root <- getwd()
raw_results_dir <- file.path(proj_root, "results", "raw")
summary_file <- file.path(proj_root, "results", "benchmark_summary_apollo_bgw_cpu.xlsx")
session_info_file <- file.path(proj_root, "results", "session_info_apollo_cpu.txt")

dir.create(raw_results_dir, recursive = TRUE, showWarnings = FALSE)

benchmark_results <- list()


In [4]:
# -------------------------------------------------------------------
# 3. Benchmark environment
# -------------------------------------------------------------------



n_cores <- 8L
logical_cpus <- parallel::detectCores(logical = TRUE)

cpu_name <- tryCatch(
  trimws(system(
    "awk -F: '/^model name/ {print $2; exit}' /proc/cpuinfo",
    intern = TRUE
  )),
  error = function(e) NA_character_
)

cpu_info <- tryCatch(
  system("lscpu", intern = TRUE),
  error = function(e) paste("lscpu failed:", conditionMessage(e))
)

cat("CPU model     :", cpu_name, "\n")
cat("Logical CPUs  :", logical_cpus, "\n")
cat("Apollo nCores :", n_cores, "\n")
cat("R version     :", R.version.string, "\n")
cat("Apollo version:", as.character(packageVersion("apollo")), "\n")

sink(session_info_file)
cat("R session information\n")
print(sessionInfo())

cat("\nPackage versions\n")
for (pkg in c("apollo", "bgw", "writexl")) {
  cat(pkg, ": ", as.character(packageVersion(pkg)), "\n", sep = "")
}

cat("\nBenchmark CPU settings\n")
cat("CPU model     :", cpu_name, "\n")
cat("Logical CPUs  :", logical_cpus, "\n")
cat("Apollo nCores :", n_cores, "\n")

cat("\nFull lscpu output\n")
cat(paste(cpu_info, collapse = "\n"), "\n")
sink()

cat("Session and hardware information saved to:", session_info_file, "\n")


CPU model     : Intel(R) Xeon(R) CPU @ 2.20GHz 
Logical CPUs  : 8 
Apollo nCores : 8 
R version     : R version 4.6.1 (2026-06-24) 
Apollo version: 0.3.9 
Session and hardware information saved to: /content/results/session_info_apollo_cpu.txt 


In [5]:
# ============================================================
# 4.1 500-draw
# ============================================================

# -------------------------------------------------------------------
# 1. Data
# -------------------------------------------------------------------

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL


# -------------------------------------------------------------------
# 2. Apollo settings
# -------------------------------------------------------------------

apollo_control <- list(
  modelName  = "apollo_mxl_swiss_500draws",
  modelDescr = "Swiss route choice mixed logit with 500 Halton draws",
  indivID    = "ID",
  mixing     = TRUE,
  nCores     = n_cores,
  outputDirectory = raw_results_dir
)

apollo_beta <- c(
  mu_log_b_tt    = -3,
  sigma_log_b_tt =  0.01,
  mu_log_b_tc    = -3,
  sigma_log_b_tc =  0.01,
  mu_log_b_hw    = -3,
  sigma_log_b_hw =  0.01,
  mu_log_b_ch    = -3,
  sigma_log_b_ch =  0.01
)

apollo_fixed <- c()

apollo_draws <- list(
  interDrawsType = "halton",
  interNDraws    = 500,
  interUnifDraws = c(),
  interNormDraws = c(
    "draws_tt",
    "draws_tc",
    "draws_hw",
    "draws_ch"
  ),
  intraDrawsType = "halton",
  intraNDraws    = 0,
  intraUnifDraws = c(),
  intraNormDraws = c()
)


# -------------------------------------------------------------------
# 3. Random coefficients
# -------------------------------------------------------------------

apollo_randCoeff <- function(apollo_beta, apollo_inputs) {

  randcoeff <- list()

  randcoeff[["b_tt"]] <- -exp(
    mu_log_b_tt +
      sigma_log_b_tt * draws_tt
  )

  randcoeff[["b_tc"]] <- -exp(
    mu_log_b_tc +
      sigma_log_b_tc * draws_tc
  )

  randcoeff[["b_hw"]] <- -exp(
    mu_log_b_hw +
      sigma_log_b_hw * draws_hw
  )

  randcoeff[["b_ch"]] <- -exp(
    mu_log_b_ch +
      sigma_log_b_ch * draws_ch
  )

  return(randcoeff)
}


# -------------------------------------------------------------------
# 4. Validate inputs
# -------------------------------------------------------------------

apollo_inputs <- apollo_validateInputs()


# -------------------------------------------------------------------
# 5. Model likelihood
# -------------------------------------------------------------------

apollo_probabilities <- function(apollo_beta,
                                 apollo_inputs,
                                 functionality = "estimate") {

  apollo_attach(apollo_beta, apollo_inputs)
  on.exit(apollo_detach(apollo_beta, apollo_inputs))

  P <- list()

  V <- list()

  V[["alt1"]] <- b_tt * tt1 +
    b_tc * tc1 +
    b_hw * hw1 +
    b_ch * ch1

  V[["alt2"]] <- b_tt * tt2 +
    b_tc * tc2 +
    b_hw * hw2 +
    b_ch * ch2

  mnl_settings <- list(
    alternatives = c(
      alt1 = 1,
      alt2 = 2
    ),
    avail = list(
      alt1 = 1,
      alt2 = 1
    ),
    choiceVar = choice,
    utilities = V
  )

  P[["model"]] <- apollo_mnl(
    mnl_settings,
    functionality
  )

  P <- apollo_panelProd(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_avgInterDraws(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_prepareProb(
    P,
    apollo_inputs,
    functionality
  )

  return(P)
}


# -------------------------------------------------------------------
# 6. Estimate model
# -------------------------------------------------------------------

estimate_settings <- list(
  printLevel     = 2)

model <- apollo_estimate(
  apollo_beta,
  apollo_fixed,
  apollo_probabilities,
  apollo_inputs,
  estimate_settings = estimate_settings
)


# -------------------------------------------------------------------
# 7. Output
# -------------------------------------------------------------------

apollo_modelOutput(model)

# -------------------------------------------------------------------
# 8. Benchmark record
# -------------------------------------------------------------------

timing_apollo <- data.frame(
  draws = 500L,
  initial_loglik = model$LLStart,
  final_loglik = model$maximum,
  pre_estimation_time_sec = model$timePre,
  estimation_time_sec = model$timeEst,
  postestimation_time_sec = model$timePost
)

benchmark_results[[as.character(500L)]] <- timing_apollo
print(timing_apollo)


apollo_draws and apollo_randCoeff were found, so apollo_control$mixing
  was set to TRUE
Several observations per individual detected based on the value of ID.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Generating inter-individual draws .... Done
Preparing user-defined functions.

Testing likelihood function...

Overview of choices for MNL model component :
                                    alt1    alt2
Times available                  3492.00 3492.00
Times chosen                     1734.00 1758.00
Percentage chosen overall          49.66   50.34
Percentage chosen when available   49.66   50.34


Pre-processing likelihood function...
Creating cluster...
Preparing workers for multithreading...

Testing influence of parameters
Starting main estimation

BGW using analytic model derivatives supplied by caller...


Iterates will be written to: 
 /content/results/raw/apollo_mxl_swiss_500draws_iterations.csv


In [6]:
# ============================================================
# 4.2 1000-draw
# ============================================================

# -------------------------------------------------------------------
# 1. Data
# -------------------------------------------------------------------

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL


# -------------------------------------------------------------------
# 2. Apollo settings
# -------------------------------------------------------------------

apollo_control <- list(
  modelName  = "apollo_mxl_swiss_1000draws",
  modelDescr = "Swiss route choice mixed logit with 1000 Halton draws",
  indivID    = "ID",
  mixing     = TRUE,
  nCores     = n_cores,
  outputDirectory = raw_results_dir
)

apollo_beta <- c(
  mu_log_b_tt    = -3,
  sigma_log_b_tt =  0.01,
  mu_log_b_tc    = -3,
  sigma_log_b_tc =  0.01,
  mu_log_b_hw    = -3,
  sigma_log_b_hw =  0.01,
  mu_log_b_ch    = -3,
  sigma_log_b_ch =  0.01
)

apollo_fixed <- c()

apollo_draws <- list(
  interDrawsType = "halton",
  interNDraws    = 1000,
  interUnifDraws = c(),
  interNormDraws = c(
    "draws_tt",
    "draws_tc",
    "draws_hw",
    "draws_ch"
  ),
  intraDrawsType = "halton",
  intraNDraws    = 0,
  intraUnifDraws = c(),
  intraNormDraws = c()
)


# -------------------------------------------------------------------
# 3. Random coefficients
# -------------------------------------------------------------------

apollo_randCoeff <- function(apollo_beta, apollo_inputs) {

  randcoeff <- list()

  randcoeff[["b_tt"]] <- -exp(
    mu_log_b_tt +
      sigma_log_b_tt * draws_tt
  )

  randcoeff[["b_tc"]] <- -exp(
    mu_log_b_tc +
      sigma_log_b_tc * draws_tc
  )

  randcoeff[["b_hw"]] <- -exp(
    mu_log_b_hw +
      sigma_log_b_hw * draws_hw
  )

  randcoeff[["b_ch"]] <- -exp(
    mu_log_b_ch +
      sigma_log_b_ch * draws_ch
  )

  return(randcoeff)
}


# -------------------------------------------------------------------
# 4. Validate inputs
# -------------------------------------------------------------------

apollo_inputs <- apollo_validateInputs()


# -------------------------------------------------------------------
# 5. Model likelihood
# -------------------------------------------------------------------

apollo_probabilities <- function(apollo_beta,
                                 apollo_inputs,
                                 functionality = "estimate") {

  apollo_attach(apollo_beta, apollo_inputs)
  on.exit(apollo_detach(apollo_beta, apollo_inputs))

  P <- list()

  V <- list()

  V[["alt1"]] <- b_tt * tt1 +
    b_tc * tc1 +
    b_hw * hw1 +
    b_ch * ch1

  V[["alt2"]] <- b_tt * tt2 +
    b_tc * tc2 +
    b_hw * hw2 +
    b_ch * ch2

  mnl_settings <- list(
    alternatives = c(
      alt1 = 1,
      alt2 = 2
    ),
    avail = list(
      alt1 = 1,
      alt2 = 1
    ),
    choiceVar = choice,
    utilities = V
  )

  P[["model"]] <- apollo_mnl(
    mnl_settings,
    functionality
  )

  P <- apollo_panelProd(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_avgInterDraws(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_prepareProb(
    P,
    apollo_inputs,
    functionality
  )

  return(P)
}


# -------------------------------------------------------------------
# 6. Estimate model
# -------------------------------------------------------------------

estimate_settings <- list(
  printLevel     = 2)

model <- apollo_estimate(
  apollo_beta,
  apollo_fixed,
  apollo_probabilities,
  apollo_inputs,
  estimate_settings = estimate_settings
)


# -------------------------------------------------------------------
# 7. Output
# -------------------------------------------------------------------

apollo_modelOutput(model)

# -------------------------------------------------------------------
# 8. Benchmark record
# -------------------------------------------------------------------

timing_apollo <- data.frame(
  draws = 1000L,
  initial_loglik = model$LLStart,
  final_loglik = model$maximum,
  pre_estimation_time_sec = model$timePre,
  estimation_time_sec = model$timeEst,
  postestimation_time_sec = model$timePost
)

benchmark_results[[as.character(1000L)]] <- timing_apollo
print(timing_apollo)


apollo_draws and apollo_randCoeff were found, so apollo_control$mixing
  was set to TRUE
Several observations per individual detected based on the value of ID.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Generating inter-individual draws .... Done
Preparing user-defined functions.

Testing likelihood function...

Overview of choices for MNL model component :
                                    alt1    alt2
Times available                  3492.00 3492.00
Times chosen                     1734.00 1758.00
Percentage chosen overall          49.66   50.34
Percentage chosen when available   49.66   50.34


Pre-processing likelihood function...
Creating cluster...
Preparing workers for multithreading...

Testing influence of parameters
Starting main estimation

BGW using analytic model derivatives supplied by caller...


Iterates will be written to: 
 /content/results/raw/apollo_mxl_swiss_1000draws_iterations.csv

In [7]:
# ============================================================
# 4.3 2000-draw
# ============================================================

# -------------------------------------------------------------------
# 1. Data
# -------------------------------------------------------------------

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL


# -------------------------------------------------------------------
# 2. Apollo settings
# -------------------------------------------------------------------

apollo_control <- list(
  modelName  = "apollo_mxl_swiss_2000draws",
  modelDescr = "Swiss route choice mixed logit with 2000 Halton draws",
  indivID    = "ID",
  mixing     = TRUE,
  nCores     = n_cores,
  outputDirectory = raw_results_dir
)

apollo_beta <- c(
  mu_log_b_tt    = -3,
  sigma_log_b_tt =  0.01,
  mu_log_b_tc    = -3,
  sigma_log_b_tc =  0.01,
  mu_log_b_hw    = -3,
  sigma_log_b_hw =  0.01,
  mu_log_b_ch    = -3,
  sigma_log_b_ch =  0.01
)

apollo_fixed <- c()

apollo_draws <- list(
  interDrawsType = "halton",
  interNDraws    = 2000,
  interUnifDraws = c(),
  interNormDraws = c(
    "draws_tt",
    "draws_tc",
    "draws_hw",
    "draws_ch"
  ),
  intraDrawsType = "halton",
  intraNDraws    = 0,
  intraUnifDraws = c(),
  intraNormDraws = c()
)


# -------------------------------------------------------------------
# 3. Random coefficients
# -------------------------------------------------------------------

apollo_randCoeff <- function(apollo_beta, apollo_inputs) {

  randcoeff <- list()

  randcoeff[["b_tt"]] <- -exp(
    mu_log_b_tt +
      sigma_log_b_tt * draws_tt
  )

  randcoeff[["b_tc"]] <- -exp(
    mu_log_b_tc +
      sigma_log_b_tc * draws_tc
  )

  randcoeff[["b_hw"]] <- -exp(
    mu_log_b_hw +
      sigma_log_b_hw * draws_hw
  )

  randcoeff[["b_ch"]] <- -exp(
    mu_log_b_ch +
      sigma_log_b_ch * draws_ch
  )

  return(randcoeff)
}


# -------------------------------------------------------------------
# 4. Validate inputs
# -------------------------------------------------------------------

apollo_inputs <- apollo_validateInputs()


# -------------------------------------------------------------------
# 5. Model likelihood
# -------------------------------------------------------------------

apollo_probabilities <- function(apollo_beta,
                                 apollo_inputs,
                                 functionality = "estimate") {

  apollo_attach(apollo_beta, apollo_inputs)
  on.exit(apollo_detach(apollo_beta, apollo_inputs))

  P <- list()

  V <- list()

  V[["alt1"]] <- b_tt * tt1 +
    b_tc * tc1 +
    b_hw * hw1 +
    b_ch * ch1

  V[["alt2"]] <- b_tt * tt2 +
    b_tc * tc2 +
    b_hw * hw2 +
    b_ch * ch2

  mnl_settings <- list(
    alternatives = c(
      alt1 = 1,
      alt2 = 2
    ),
    avail = list(
      alt1 = 1,
      alt2 = 1
    ),
    choiceVar = choice,
    utilities = V
  )

  P[["model"]] <- apollo_mnl(
    mnl_settings,
    functionality
  )

  P <- apollo_panelProd(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_avgInterDraws(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_prepareProb(
    P,
    apollo_inputs,
    functionality
  )

  return(P)
}


# -------------------------------------------------------------------
# 6. Estimate model
# -------------------------------------------------------------------

estimate_settings <- list(
  printLevel     = 2)

model <- apollo_estimate(
  apollo_beta,
  apollo_fixed,
  apollo_probabilities,
  apollo_inputs,
  estimate_settings = estimate_settings
)


# -------------------------------------------------------------------
# 7. Output
# -------------------------------------------------------------------

apollo_modelOutput(model)

# -------------------------------------------------------------------
# 8. Benchmark record
# -------------------------------------------------------------------

timing_apollo <- data.frame(
  draws = 2000L,
  initial_loglik = model$LLStart,
  final_loglik = model$maximum,
  pre_estimation_time_sec = model$timePre,
  estimation_time_sec = model$timeEst,
  postestimation_time_sec = model$timePost
)

benchmark_results[[as.character(2000L)]] <- timing_apollo
print(timing_apollo)


apollo_draws and apollo_randCoeff were found, so apollo_control$mixing
  was set to TRUE
Several observations per individual detected based on the value of ID.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Generating inter-individual draws .... Done
Preparing user-defined functions.

Testing likelihood function...

Overview of choices for MNL model component :
                                    alt1    alt2
Times available                  3492.00 3492.00
Times chosen                     1734.00 1758.00
Percentage chosen overall          49.66   50.34
Percentage chosen when available   49.66   50.34


Pre-processing likelihood function...
Creating cluster...
Preparing workers for multithreading...

Testing influence of parameters
Starting main estimation

BGW using analytic model derivatives supplied by caller...


Iterates will be written to: 
 /content/results/raw/apollo_mxl_swiss_2000draws_iterations.csv

In [8]:
# ============================================================
# 4.4 5000-draw
# ============================================================

# -------------------------------------------------------------------
# 1. Data
# -------------------------------------------------------------------

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL


# -------------------------------------------------------------------
# 2. Apollo settings
# -------------------------------------------------------------------

apollo_control <- list(
  modelName  = "apollo_mxl_swiss_5000draws",
  modelDescr = "Swiss route choice mixed logit with 5000 Halton draws",
  indivID    = "ID",
  mixing     = TRUE,
  nCores     = n_cores,
  outputDirectory = raw_results_dir
)

apollo_beta <- c(
  mu_log_b_tt    = -3,
  sigma_log_b_tt =  0.01,
  mu_log_b_tc    = -3,
  sigma_log_b_tc =  0.01,
  mu_log_b_hw    = -3,
  sigma_log_b_hw =  0.01,
  mu_log_b_ch    = -3,
  sigma_log_b_ch =  0.01
)

apollo_fixed <- c()

apollo_draws <- list(
  interDrawsType = "halton",
  interNDraws    = 5000,
  interUnifDraws = c(),
  interNormDraws = c(
    "draws_tt",
    "draws_tc",
    "draws_hw",
    "draws_ch"
  ),
  intraDrawsType = "halton",
  intraNDraws    = 0,
  intraUnifDraws = c(),
  intraNormDraws = c()
)


# -------------------------------------------------------------------
# 3. Random coefficients
# -------------------------------------------------------------------

apollo_randCoeff <- function(apollo_beta, apollo_inputs) {

  randcoeff <- list()

  randcoeff[["b_tt"]] <- -exp(
    mu_log_b_tt +
      sigma_log_b_tt * draws_tt
  )

  randcoeff[["b_tc"]] <- -exp(
    mu_log_b_tc +
      sigma_log_b_tc * draws_tc
  )

  randcoeff[["b_hw"]] <- -exp(
    mu_log_b_hw +
      sigma_log_b_hw * draws_hw
  )

  randcoeff[["b_ch"]] <- -exp(
    mu_log_b_ch +
      sigma_log_b_ch * draws_ch
  )

  return(randcoeff)
}


# -------------------------------------------------------------------
# 4. Validate inputs
# -------------------------------------------------------------------

apollo_inputs <- apollo_validateInputs()


# -------------------------------------------------------------------
# 5. Model likelihood
# -------------------------------------------------------------------

apollo_probabilities <- function(apollo_beta,
                                 apollo_inputs,
                                 functionality = "estimate") {

  apollo_attach(apollo_beta, apollo_inputs)
  on.exit(apollo_detach(apollo_beta, apollo_inputs))

  P <- list()

  V <- list()

  V[["alt1"]] <- b_tt * tt1 +
    b_tc * tc1 +
    b_hw * hw1 +
    b_ch * ch1

  V[["alt2"]] <- b_tt * tt2 +
    b_tc * tc2 +
    b_hw * hw2 +
    b_ch * ch2

  mnl_settings <- list(
    alternatives = c(
      alt1 = 1,
      alt2 = 2
    ),
    avail = list(
      alt1 = 1,
      alt2 = 1
    ),
    choiceVar = choice,
    utilities = V
  )

  P[["model"]] <- apollo_mnl(
    mnl_settings,
    functionality
  )

  P <- apollo_panelProd(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_avgInterDraws(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_prepareProb(
    P,
    apollo_inputs,
    functionality
  )

  return(P)
}


# -------------------------------------------------------------------
# 6. Estimate model
# -------------------------------------------------------------------

estimate_settings <- list(
  printLevel     = 2)

model <- apollo_estimate(
  apollo_beta,
  apollo_fixed,
  apollo_probabilities,
  apollo_inputs,
  estimate_settings = estimate_settings
)


# -------------------------------------------------------------------
# 7. Output
# -------------------------------------------------------------------

apollo_modelOutput(model)

# -------------------------------------------------------------------
# 8. Benchmark record
# -------------------------------------------------------------------

timing_apollo <- data.frame(
  draws = 5000L,
  initial_loglik = model$LLStart,
  final_loglik = model$maximum,
  pre_estimation_time_sec = model$timePre,
  estimation_time_sec = model$timeEst,
  postestimation_time_sec = model$timePost
)

benchmark_results[[as.character(5000L)]] <- timing_apollo
print(timing_apollo)


apollo_draws and apollo_randCoeff were found, so apollo_control$mixing
  was set to TRUE
Several observations per individual detected based on the value of ID.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Generating inter-individual draws .... Done
Preparing user-defined functions.

Testing likelihood function...

Overview of choices for MNL model component :
                                    alt1    alt2
Times available                  3492.00 3492.00
Times chosen                     1734.00 1758.00
Percentage chosen overall          49.66   50.34
Percentage chosen when available   49.66   50.34


Pre-processing likelihood function...
Creating cluster...
Preparing workers for multithreading...

Testing influence of parameters
Starting main estimation

BGW using analytic model derivatives supplied by caller...


Iterates will be written to: 
 /content/results/raw/apollo_mxl_swiss_5000draws_iterations.csv

In [9]:
# ============================================================
# 4.5 10000-draw
# ============================================================

# -------------------------------------------------------------------
# 1. Data
# -------------------------------------------------------------------

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL


# -------------------------------------------------------------------
# 2. Apollo settings
# -------------------------------------------------------------------

apollo_control <- list(
  modelName  = "apollo_mxl_swiss_10000draws",
  modelDescr = "Swiss route choice mixed logit with 10000 Halton draws",
  indivID    = "ID",
  mixing     = TRUE,
  nCores     = n_cores,
  outputDirectory = raw_results_dir
)

apollo_beta <- c(
  mu_log_b_tt    = -3,
  sigma_log_b_tt =  0.01,
  mu_log_b_tc    = -3,
  sigma_log_b_tc =  0.01,
  mu_log_b_hw    = -3,
  sigma_log_b_hw =  0.01,
  mu_log_b_ch    = -3,
  sigma_log_b_ch =  0.01
)

apollo_fixed <- c()

apollo_draws <- list(
  interDrawsType = "halton",
  interNDraws    = 10000,
  interUnifDraws = c(),
  interNormDraws = c(
    "draws_tt",
    "draws_tc",
    "draws_hw",
    "draws_ch"
  ),
  intraDrawsType = "halton",
  intraNDraws    = 0,
  intraUnifDraws = c(),
  intraNormDraws = c()
)


# -------------------------------------------------------------------
# 3. Random coefficients
# -------------------------------------------------------------------

apollo_randCoeff <- function(apollo_beta, apollo_inputs) {

  randcoeff <- list()

  randcoeff[["b_tt"]] <- -exp(
    mu_log_b_tt +
      sigma_log_b_tt * draws_tt
  )

  randcoeff[["b_tc"]] <- -exp(
    mu_log_b_tc +
      sigma_log_b_tc * draws_tc
  )

  randcoeff[["b_hw"]] <- -exp(
    mu_log_b_hw +
      sigma_log_b_hw * draws_hw
  )

  randcoeff[["b_ch"]] <- -exp(
    mu_log_b_ch +
      sigma_log_b_ch * draws_ch
  )

  return(randcoeff)
}


# -------------------------------------------------------------------
# 4. Validate inputs
# -------------------------------------------------------------------

apollo_inputs <- apollo_validateInputs()


# -------------------------------------------------------------------
# 5. Model likelihood
# -------------------------------------------------------------------

apollo_probabilities <- function(apollo_beta,
                                 apollo_inputs,
                                 functionality = "estimate") {

  apollo_attach(apollo_beta, apollo_inputs)
  on.exit(apollo_detach(apollo_beta, apollo_inputs))

  P <- list()

  V <- list()

  V[["alt1"]] <- b_tt * tt1 +
    b_tc * tc1 +
    b_hw * hw1 +
    b_ch * ch1

  V[["alt2"]] <- b_tt * tt2 +
    b_tc * tc2 +
    b_hw * hw2 +
    b_ch * ch2

  mnl_settings <- list(
    alternatives = c(
      alt1 = 1,
      alt2 = 2
    ),
    avail = list(
      alt1 = 1,
      alt2 = 1
    ),
    choiceVar = choice,
    utilities = V
  )

  P[["model"]] <- apollo_mnl(
    mnl_settings,
    functionality
  )

  P <- apollo_panelProd(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_avgInterDraws(
    P,
    apollo_inputs,
    functionality
  )

  P <- apollo_prepareProb(
    P,
    apollo_inputs,
    functionality
  )

  return(P)
}


# -------------------------------------------------------------------
# 6. Estimate model
# -------------------------------------------------------------------

estimate_settings <- list(
  printLevel     = 2)

model <- apollo_estimate(
  apollo_beta,
  apollo_fixed,
  apollo_probabilities,
  apollo_inputs,
  estimate_settings = estimate_settings
)


# -------------------------------------------------------------------
# 7. Output
# -------------------------------------------------------------------

apollo_modelOutput(model)

# -------------------------------------------------------------------
# 8. Benchmark record
# -------------------------------------------------------------------

timing_apollo <- data.frame(
  draws = 10000L,
  initial_loglik = model$LLStart,
  final_loglik = model$maximum,
  pre_estimation_time_sec = model$timePre,
  estimation_time_sec = model$timeEst,
  postestimation_time_sec = model$timePost
)

benchmark_results[[as.character(10000L)]] <- timing_apollo
print(timing_apollo)


apollo_draws and apollo_randCoeff were found, so apollo_control$mixing
  was set to TRUE
Several observations per individual detected based on the value of ID.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Generating inter-individual draws .... Done
Preparing user-defined functions.

Testing likelihood function...

Overview of choices for MNL model component :
                                    alt1    alt2
Times available                  3492.00 3492.00
Times chosen                     1734.00 1758.00
Percentage chosen overall          49.66   50.34
Percentage chosen when available   49.66   50.34


Pre-processing likelihood function...
Creating cluster...
Preparing workers for multithreading...

Testing influence of parameters
Starting main estimation

BGW using analytic model derivatives supplied by caller...


Iterates will be written to: 
 /content/results/raw/apollo_mxl_swiss_10000draws_iterations.cs

In [10]:
# ============================================================
# 5. Summarise and export results
# ============================================================

if (!length(benchmark_results)) {
  stop("Run at least one Section 4 draw cell before exporting results.")
}

benchmark_timings <- do.call(rbind, benchmark_results)
row.names(benchmark_timings) <- NULL

benchmark_summary <- data.frame(
  package = "Apollo",
  implementation = "apollo_bgw_cpu",
  backend = "CPU",
  device = cpu_name,
  optimiser = "BGW",
  draws = benchmark_timings$draws,
  n_cores = n_cores,
  logical_cpus = logical_cpus,
  initial_loglik = benchmark_timings$initial_loglik,
  final_loglik = benchmark_timings$final_loglik,
  pre_estimation_time_sec = benchmark_timings$pre_estimation_time_sec,
  estimation_time_sec = benchmark_timings$estimation_time_sec,
  postestimation_time_sec = benchmark_timings$postestimation_time_sec,
  stringsAsFactors = FALSE
)

benchmark_summary <- benchmark_summary[order(benchmark_summary$draws), ]
row.names(benchmark_summary) <- NULL

time_columns <- c(
  "pre_estimation_time_sec",
  "estimation_time_sec",
  "postestimation_time_sec"
)
benchmark_summary[time_columns] <- lapply(
  benchmark_summary[time_columns],
  round,
  digits = 4
)
benchmark_summary[c("initial_loglik", "final_loglik")] <- lapply(
  benchmark_summary[c("initial_loglik", "final_loglik")],
  round,
  digits = 2
)

writexl::write_xlsx(
  list(benchmark_summary = benchmark_summary),
  path = summary_file
)

print(benchmark_summary)
cat("Excel file saved to:", summary_file, "\n")


  package implementation backend                         device optimiser draws
1  Apollo apollo_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW   500
2  Apollo apollo_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW  1000
3  Apollo apollo_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW  2000
4  Apollo apollo_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW  5000
5  Apollo apollo_bgw_cpu     CPU Intel(R) Xeon(R) CPU @ 2.20GHz       BGW 10000
  n_cores logical_cpus initial_loglik final_loglik pre_estimation_time_sec
1       8            8       -2253.78     -1445.69                 32.8331
2       8            8       -2253.78     -1445.08                 40.0639
3       8            8       -2253.78     -1444.09                 63.5047
4       8            8       -2253.78     -1444.67                114.7776
5       8            8       -2253.78     -1444.56                195.0033
  estimation_time_sec postestimation_time_sec
1             15.6230   